In [113]:
import torch
import torch.nn as nn
import torch.optim as op
from torchvision import  datasets, transforms,models
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import numpy as np

In [114]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device", device)

Using device cuda


In [115]:
train_transform = transforms.Compose([
  transforms.Resize((224,224)),
  transforms.RandomHorizontalFlip(), # Data augumentation steps
  transforms.RandomRotation(20),     # Data Augumentation step
  transforms.ToTensor(),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [116]:
# Loading the dataset 
train_dataset = datasets.ImageFolder("D:/archive/chest_xray/train", transform= train_transform)
test_dataset = datasets.ImageFolder("D:/archive/chest_xray/test", transform= test_transform)


In [117]:
# handle Imbalanced class
labels = [label for _, label in train_dataset.samples]

class_counts = [labels.count(0), labels.count(1)]  # 0 - Normal , 1 - Pneumonia
total_samples = sum(class_counts)
class_weights = [total_samples/c for c in class_counts] # this is inverse freqency
sample_weights = [class_weights[label] for label in labels] # storing inverse frequency

sampler = WeightedRandomSampler(weights = sample_weights, num_samples=len(sample_weights), replacement=True)

In [118]:
train_loader = DataLoader(train_dataset,batch_size=16, sampler=sampler,num_workers=4, pin_memory=True)


In [119]:
test_loader = DataLoader(test_dataset, batch_size=16 , shuffle= False, num_workers=2, pin_memory=True)

In [120]:
# Load Resnet 50
model = models.resnet50(pretrained = True)
num_flts = model.fc.in_features
model.fc=  nn.Linear(num_flts,2)
model = model.to(device)

C:\Users\Ayush Keshri\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Ayush Keshri\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [121]:
weights = torch.tensor([ class_counts[1]/class_counts[0],1.0]).to(device)
criteria = nn.CrossEntropyLoss(weight=weights)
optimizer = op.Adam(model.parameters(),lr =0.0001)

In [123]:
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss =0
    for images, labels in train_loader:
        images , labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criteria(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")

Epoch [1/10], Loss: 0.1120
Epoch [2/10], Loss: 0.0715
Epoch [3/10], Loss: 0.0380
Epoch [4/10], Loss: 0.0362
Epoch [5/10], Loss: 0.0451
Epoch [6/10], Loss: 0.0375
Epoch [7/10], Loss: 0.0307
Epoch [8/10], Loss: 0.0254
Epoch [9/10], Loss: 0.0278
Epoch [10/10], Loss: 0.0214


In [124]:
all_labels, all_preds, all_probs = [], [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images , labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = nn.Softmax(dim=1)(outputs)
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:,1].cpu().numpy())
        
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)

print("Accuracy using Resnet-50 is  ", accuracy)
print("Precision using Resnet-50 is  ",precision)
print("Recall using Resnet-50 is  ", recall)
print("AUC using Resnet-50 is  ", auc)


Accuracy using Resnet-50 is   0.8557692307692307
Precision using Resnet-50 is   0.8164556962025317
Recall using Resnet-50 is   0.9923076923076923
AUC using Resnet-50 is   0.974364453210607


In [127]:
from PIL import Image
import torch
from torchvision import transforms, models

In [128]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [129]:
model = models.resnet50(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, 2)  # 2 classes
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()

C:\Users\Ayush Keshri\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Ayush Keshri\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
C:\Users\Ayush Keshri\AppData\Local\Temp\ipykernel_53204\1788699240.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more d

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [130]:
transform = transforms.Compose([
 transforms.Resize((224, 224)),
 transforms.ToTensor(),
 transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [136]:
img_path = "D:/archive/chest_xray/test/NORMAL/IM-0061-0001.jpeg"
image = Image.open(img_path).convert("RGB")
image = transform(image)             # [3,224,224]
image = image.unsqueeze(0).to(device)

In [137]:
with torch.no_grad():
    outputs = model(image)                   # logits [1,2]
    probs = torch.softmax(outputs, dim=1)   # probabilities
    pred_class = torch.argmax(outputs, dim=1) 

In [138]:
class_names = ["Normal", "Pneumonia"]
print("Predicted class:", class_names[pred_class.item()])
print(f"Confidence: {probs[0, pred_class].item()*100:.2f}%")

Predicted class: Normal
Confidence: 100.00%


In [139]:
from sklearn.metrics import confusion_matrix

all_labels, all_preds = [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[147  87]
 [  3 387]]


In [126]:
torch.save(model.state_dict(), "best_model.pth")